# 03. WAQI Live Integration

**Inspect live air quality data using the WAQI API**

Use this notebook to:
- validate WAQI token setup
- inspect live station and city responses
- sanity-check WAQI-side behavior before or alongside manual station curation

Current preferred workflow:
- manual CSV downloads are the primary expansion source
- this notebook is now optional and mostly for live inspection or debugging

Recommended flow:
- upstream: `01_Global_Forecast_Error_Reduction_Plan.ipynb`
- downstream: `04_Station_Expansion_Eval.ipynb` only when you want WAQI-side context in addition to manual station files


## [INFO] Step 1: Load Libraries

In [7]:
import sys
from pathlib import Path

# Find the project root directory
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / 'src'))

import pandas as pd
import plotly.graph_objects as go
from datetime import datetime

print(f"[INFO] Project root: {project_root}")
print(f"[OK] Libraries loaded")

[INFO] Project root: c:\Users\rbeyz\Desktop\AirPulse_Global
[OK] Libraries loaded


## Step 2: WAQI Token Setup

**How to get a token:**
1. Go to https://aqicn.org/data-platform/token/
2. Get a free token via email
3. Create an `api_token.txt` file in the project root
4. Paste the token into the file (token only, nothing else!)

In [8]:
from airpulse.waqi_integration import WAQIClient

try:
    client = WAQIClient()
    print("[OK] WAQI API token loaded successfully")
    print("   Masked token: " + client.token[:4] + "****" + client.token[-4:])
except ValueError as e:
    print(f"[ERROR] Token not found: {e}")

[OK] WAQI API token loaded successfully
   Masked token: de10****9052


## Step 3: Live Istanbul Data

In [9]:
# Istanbul city feed
istanbul_feed = client.get_city_feed('istanbul')

if istanbul_feed:
    formatted = client.format_for_display(istanbul_feed)
    
    print("Live Air Quality in Istanbul")
    print("=" * 50)
    print(f"AQI: {formatted['aqi']} ({formatted['level']})")
    print(f"Dominant Pollutant: {formatted['dominentpol']}")
    print(f"Time: {formatted['time']}")
    print(f"\nPollutants:")
    for pol, val in formatted['pollutants'].items():
        print(f"  {pol}: {val}")
else:
    print("[ERROR] Could not retrieve Istanbul data")

Live Air Quality in Istanbul
AQI: 66 (Moderate)
Dominant Pollutant: PM25
Time: 2026-05-07 09:00:00

Pollutants:
  PM2.5: 66
  PM10: 39
  NO2: 32.5
  SO2: 3.8
  O3: 1.8
  CO: 7.2


## Step 4: Search Istanbul Stations

In [10]:
# Search Istanbul stations
stations = client.search_stations('istanbul')

print(f"Number of stations found: {len(stations)}\n")

# Show the first 10 stations
station_list = []

for i, station in enumerate(stations[:10], 1):
    name = station.get('station', {}).get('name', 'Unknown')
    aqi = station.get('aqi', 'N/A')
    uid = station.get('uid', 'N/A')
    
    station_list.append({
        'No': i,
        'Name': name,
        'AQI': aqi,
        'UID': uid
    })
    
    print(f"{i:2d}. {name:40s} | AQI: {aqi:5s} | UID: {uid}")

# Display as a DataFrame
stations_df = pd.DataFrame(station_list)
display(stations_df)

Number of stations found: 21

 1. Mobil 2, Turkey                          | AQI: 80    | UID: 14843
 2. Istanbul (Aksaray), Turkey               | AQI: 66    | UID: 4143
 3. Selimiye, Turkey                         | AQI: 64    | UID: 8773
 4. Istanbul Kagithane, Turkey               | AQI: 64    | UID: 8154
 5. Catladikapi, Turkey                      | AQI: 62    | UID: 8772
 6. Istanbul (Esenler), Turkey               | AQI: 60    | UID: 4146
 7. Istanbul (Kagithane), Turkey             | AQI: 60    | UID: 5382
 8. Istanbul (Kadikoy), Turkey               | AQI: 59    | UID: 4147
 9. Istanbul (Alibeykoy), Turkey             | AQI: 51    | UID: 4144
10. Istanbul (Besiktas), Turkey              | AQI: 44    | UID: 4145


,No,Name,AQI,UID
0,1,"Mobil 2, Turkey",80,14843
1,2,"Istanbul (Aksaray), Turkey",66,4143
2,3,"Selimiye, Turkey",64,8773
3,4,"Istanbul Kagithane, Turkey",64,8154
4,5,"Catladikapi, Turkey",62,8772
5,6,"Istanbul (Esenler), Turkey",60,4146
6,7,"Istanbul (Kagithane), Turkey",60,5382
7,8,"Istanbul (Kadikoy), Turkey",59,4147
8,9,"Istanbul (Alibeykoy), Turkey",51,4144
9,10,"Istanbul (Besiktas), Turkey",44,4145


## Step 5: Show Stations on the Map

In [17]:
import plotly.io as pio

# Prepare map data
map_data = []

for station in stations[:20]:
    geo = station.get('station', {}).get('geo', [])
    if len(geo) >= 2:
        aqi = station.get('aqi', 'N/A')

        try:
            aqi_val = int(aqi) if aqi != '-' else 0
        except Exception:
            aqi_val = 0

        if aqi_val <= 50:
            color = '#00E400'
        elif aqi_val <= 100:
            color = '#FFFF00'
        elif aqi_val <= 150:
            color = '#FF7E00'
        elif aqi_val <= 200:
            color = '#FF0000'
        elif aqi_val <= 300:
            color = '#8F3F97'
        else:
            color = '#7E0023'

        map_data.append({
            'lat': geo[0],
            'lon': geo[1],
            'name': station.get('station', {}).get('name', 'Unknown'),
            'aqi': aqi,
            'color': color,
        })

fig = go.Figure()

for station in map_data:
    fig.add_trace(go.Scattermapbox(
        lat=[station['lat']],
        lon=[station['lon']],
        mode='markers',
        marker=dict(size=15, color=station['color'], opacity=0.8),
        text=[f"<b>{station['name']}</b><br>AQI: {station['aqi']}"],
        hoverinfo='text',
        showlegend=False
    ))

fig.update_layout(
    mapbox=dict(
        style="open-street-map",
        center=dict(lat=41.0082, lon=28.9784),
        zoom=9
    ),
    height=600,
    title="Istanbul WAQI Stations"
)

pio.renderers.default = "browser"
fig.show()

print(f"\nShowing {len(map_data)} stations on the map")


C:\Users\rbeyz\AppData\Local\Temp\ipykernel_26300\3387452792.py:40: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(



Showing 20 stations on the map


##  Step 6: Detailed Station Data

In [19]:
# Select a station and retrieve its details
if stations:
    # Get the UID of the first station
    first_station_uid = stations[0].get('uid')
    first_station_name = stations[0].get('station', {}).get('name', 'Unknown')
    
    print(f"Detailed data: {first_station_name} (UID: {first_station_uid})\n")
    
    # Fetch the feed
    feed = client.get_station_by_id(first_station_uid)
    
    if feed:
        formatted = client.format_for_display(feed)
        
        print(f"AQI: {formatted['aqi']} ({formatted['level']})")
        print(f"Dominant: {formatted['dominentpol']}")
        print(f"Time: {formatted['time']}")
        print(f"Location: {formatted['latitude']}, {formatted['longitude']}")
        print(f"\nPollutants:")
        
        for pol, val in formatted['pollutants'].items():
            print(f"  {pol:6s}: {val}")
        
        print(f"\nAttributions:")
        for attr in formatted['attributions']:
            print(f"  - {attr}")

Detailed data: Mobil 2, Turkey (UID: 14843)

AQI: 80 (Moderate)
Dominant: PM25
Time: 2026-05-07 09:00:00
Location: 41.051663093475156, 28.867944221876538

Pollutants:
  PM2.5 : 80
  PM10  : 39
  NO2   : 5.4
  SO2   : 2.9
  O3    : 14.1
  CO    : 13

Attributions:
  - Turkey National Air Quality Monitoring Network (Ulusal Hava Kalitesi İzleme Ağı) (https://sim.csb.gov.tr/Services/AirQuality)
  - World Air Quality Index Project (https://waqi.info/)


## Summary

In this notebook:
- WAQI API token setup
- Fetching live Istanbul data
- Searching stations
- Visualizing them on a map
- Retrieving detailed station data

**Next steps:**
1. Compare with offline forecasts
2. Use it in the Streamlit app
3. Real-time monitoring